In [ ]:
import re
from difflib import SequenceMatcher
from typing import Optional


In [ ]:
"""
Skill Normalization Engine — built from clean_jobs_descriptions_combined.csv

Data-driven: the canonical vocabulary and alias list below were derived by
mining the actual 'skills' and 'extracted_skills' columns of the dataset,
not hand-invented examples.
"""

import re
import pandas as pd
from collections import Counter
from difflib import SequenceMatcher
from typing import Optional


class SkillNormalizer:
    def __init__(self, fuzzy_threshold: float = 0.88):
        self.fuzzy_threshold = fuzzy_threshold

        # Canonical skill -> known aliases (lowercased).
        # Base 26 = actual vocabulary found in this dataset's extracted_skills column.
        # Extras = additional variants found in the raw 'skills' text column
        # (not yet in the extraction vocabulary, but worth normalizing if they appear).
        self.canonical_map: dict[str, list[str]] = {
            "SQL":              ["sql", "structured query language"],
            "HTML":             ["html", "html5"],
            "CSS":              ["css", "css3"],
            "Python":           ["python", "python3", "python 3", "py"],
            "JavaScript":       ["javascript", "java script", "js"],
            "Java":             ["java"],
            "AWS":              ["aws", "amazon web services"],
            "Azure":            ["azure", "microsoft azure"],
            "Tableau":          ["tableau"],
            "React":            ["react", "reactjs", "react.js"],
            "Power BI":         ["power bi", "powerbi", "power-bi"],
            "Angular":          ["angular", "angularjs", "angular.js"],
            "Excel":            ["excel", "ms excel", "microsoft excel"],
            "Oracle":           ["oracle"],
            "R":                ["r", "r programming"],
            "Spark":            ["spark", "apache spark"],
            "Hadoop":           ["hadoop"],
            "MySQL":            ["mysql", "my sql"],
            "Git":              ["git"],
            "GCP":              ["gcp", "google cloud", "google cloud platform"],
            "Docker":           ["docker"],
            "Kubernetes":       ["kubernetes", "k8s"],
            "Scikit-learn":     ["scikit-learn", "scikit learn", "sklearn"],
            "TensorFlow":       ["tensorflow", "tensor flow"],
            "PyTorch":          ["pytorch", "py torch"],
            "MongoDB":          ["mongodb", "mongo db"],
            # Found in raw 'skills' text but not yet in extraction vocabulary:
            "Node.js":          ["node.js", "nodejs", "node js"],
            "Vue.js":           ["vue.js", "vuejs", "vue"],
            "TypeScript":       ["typescript", "type script", "ts"],
            "C++":              ["c++", "cpp"],
            "C#":               ["c#", "c sharp", "csharp"],
            ".NET":             [".net", "dot net", ".net core"],
            "PostgreSQL":       ["postgresql", "postgres", "postgre sql"],
            "Machine Learning": ["machine learning", "ml"],
            "Artificial Intelligence": ["artificial intelligence", "ai"],
        }

        self._alias_lookup: dict[str, str] = {}
        for canonical, aliases in self.canonical_map.items():
            for alias in aliases:
                self._alias_lookup[alias] = canonical

        self.unmatched_log: list[str] = []

    # ---------- Public API ----------

    def normalize(self, raw_skill: str) -> str:
        if not raw_skill or not str(raw_skill).strip():
            return ""

        cleaned = self._clean(raw_skill)

        if cleaned in self._alias_lookup:
            return self._alias_lookup[cleaned]

        best_match = self._fuzzy_match(cleaned)
        if best_match:
            return best_match

        self.unmatched_log.append(raw_skill.strip())
        return self._title_case_fallback(raw_skill)

    def normalize_list(self, raw_skills: list[str]) -> list[str]:
        seen = set()
        result = []
        for skill in raw_skills:
            norm = self.normalize(skill)
            if norm and norm not in seen:
                seen.add(norm)
                result.append(norm)
        return result

    def normalize_extracted_skills_field(self, field_value: str) -> list[str]:
        """Parse this dataset's 'extracted_skills' format:
        'JavaScript (Programming), HTML (Programming), ...' -> normalized list.
        Strips the trailing '(Category)' tag before normalizing.
        """
        if not isinstance(field_value, str) or not field_value.strip():
            return []
        parts = [p.strip() for p in field_value.split(",") if p.strip()]
        names = [re.sub(r"\s*\([^)]*\)\s*$", "", p).strip() for p in parts]
        return self.normalize_list(names)

    def add_alias(self, canonical: str, alias: str) -> None:
        alias_clean = self._clean(alias)
        self._alias_lookup[alias_clean] = canonical
        self.canonical_map.setdefault(canonical, [])
        if alias_clean not in self.canonical_map[canonical]:
            self.canonical_map[canonical].append(alias_clean)

    # ---------- Internal helpers ----------

    @staticmethod
    def _clean(text: str) -> str:
        text = str(text).strip().lower()
        text = re.sub(r"[_/]", " ", text)
        text = re.sub(r"\s+", " ", text)
        text = text.strip(" .,-")
        return text

    @staticmethod
    def _title_case_fallback(raw: str) -> str:
        cleaned = re.sub(r"\s+", " ", str(raw).strip())
        if re.search(r"[+#.]", cleaned):
            return cleaned
        return cleaned.title()

    def _fuzzy_match(self, cleaned: str) -> Optional[str]:
        best_score, best_canonical = 0.0, None
        for alias, canonical in self._alias_lookup.items():
            score = SequenceMatcher(None, cleaned, alias).ratio()
            if score > best_score:
                best_score, best_canonical = score, canonical
        return best_canonical if best_score >= self.fuzzy_threshold else None


# ---------------- Run against the real dataset ----------------
if __name__ == "__main__":
    df = pd.read_csv("/content/clean_jobs_descriptions_combined (1).csv")
    normalizer = SkillNormalizer()

    # BEFORE: raw variant counts (proves the casing-inconsistency problem is real)
    raw_counter = Counter()
    for val in df["extracted_skills"].dropna():
        for p in val.split(","):
            name = re.sub(r"\s*\([^)]*\)\s*$", "", p.strip()).strip()
            if name:
                raw_counter[name] += 1

    print("=== BEFORE normalization (raw surface forms) ===")
    for skill, cnt in raw_counter.most_common(10):
        print(f"{cnt:5d}  {skill!r}")

    # AFTER: apply normalizer, add a new column
    df["normalized_skills"] = df["extracted_skills"].apply(
        normalizer.normalize_extracted_skills_field
    )

    after_counter = Counter()
    for skills in df["normalized_skills"]:
        after_counter.update(skills)

    print("\n=== AFTER normalization (canonical forms) ===")
    for skill, cnt in after_counter.most_common(10):
        print(f"{cnt:5d}  {skill!r}")

    print("\n=== Proof of merge (previously split, now unified) ===")
    for canon in ["React", "Machine Learning", "Scikit-learn"]:
        print(f"{canon:20s} -> {after_counter[canon]} total mentions after merge")

    if normalizer.unmatched_log:
        print(f"\n{len(normalizer.unmatched_log)} unmatched raw strings logged for review.")
    else:
        print("\nNo unmatched skills in extracted_skills column — full coverage.")

    df.to_csv("/content/jobs_with_normalized_skills.csv", index=False)
    print("\nSaved: /content/jobs_with_normalized_skills.csv")

=== BEFORE normalization (raw surface forms) ===
  712  'SQL'
  398  'HTML'
  398  'CSS'
  360  'Python'
  249  'JavaScript'
  245  'Java'
  130  'AWS'
  130  'Azure'
  127  'Tableau'
  112  'React'

=== AFTER normalization (canonical forms) ===
  360  'Python'
  356  'SQL'
  249  'JavaScript'
  245  'Java'
  199  'HTML'
  199  'CSS'
  130  'AWS'
  130  'Azure'
  127  'Tableau'
  112  'React'

=== Proof of merge (previously split, now unified) ===
React                -> 112 total mentions after merge
Machine Learning     -> 0 total mentions after merge
Scikit-learn         -> 37 total mentions after merge

No unmatched skills in extracted_skills column — full coverage.

Saved: /content/jobs_with_normalized_skills.csv
